In [ ]:
import warnings  # Standard library module used to control how warning messages are displayed

# Ignore all warnings
warnings.filterwarnings("ignore")  # Suppress all warnings so notebook output stays clean (e.g. deprecation notices from HF libraries)


# Phase 1: Data Inspection


In [ ]:
from huggingface_hub import hf_hub_download  # Utility to download a single file from a Hugging Face Hub repo (dataset/model) into local cache
import zipfile, os  # zipfile to extract the downloaded archive, os for filesystem path/listing operations

# Download the Hindi NER dataset ZIP file from Hugging Face
z = hf_hub_download(  # Downloads (or reuses the cached copy of) the file and returns its local path
    "ai4bharat/naamapadam",  # Hugging Face dataset repository
    "data/hi_IndicNER_v1.0.zip",  # Path of the ZIP file inside the repository
    repo_type="dataset",  # Specify that the repository contains a dataset
)  # End of the hf_hub_download(...) call

# Define the directory where the extracted files will be stored
d = z + ".x"  # Reuse the zip's cache path with a ".x" suffix as the extraction target directory

# Extract the ZIP file only if the extraction directory doesn't already exist
if not os.path.isdir(d):  # Guard so repeated notebook runs don't re-extract every time
    with zipfile.ZipFile(z) as f:  # Open the downloaded zip archive for reading
        f.extractall(d)  # Extract all files into the specified directory

# Display the names of files and folders inside the extraction directory
print(os.listdir(d))  # Sanity-check which files (train/val/test JSON) are now available on disk


['hi_test.json', 'hi_train.json', 'hi_val.json']


In [ ]:
import json  # Standard library module to parse JSON Lines records from the dataset files

# Construct the path to the Hindi training dataset JSON file
train = os.path.join(d, "hi_train.json")  # Build the full path to the training split inside the extracted directory

# Open the JSON file using UTF-8 encoding
with open(train, encoding="utf-8") as f:  # UTF-8 is required because the data contains Devanagari script
    # Read only the first line of the file
    first = f.readline()  # The file is JSON Lines format, so each line is one standalone JSON record

# Print the first line to inspect the dataset format
print(first)  # Shows the raw structure (expects keys like "words" and "ner") before writing a full loader


{"words": ["सेक्टर", "55/56", "के", "एसएचओ", "अरविंद", "कुमार", "ने", "बताया", "कि", "इस", "मामले", "में", "आईपीसी", "की", "धारा", "376", "-", "डी", "(", "गैंगरेप", ")", "के", "तहत", "मामला", "दर्ज", "कर", "लिया", "गया", "है", "।"], "ner": ["O", "O", "O", "O", "B-PER", "I-PER", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"]}



In [ ]:
# Define a function to load the dataset from a JSONL file
def load(path):  # path: location of one split's .json file (actually JSON Lines, one record per line)
    rows = []  # Will accumulate (words, ner_tags) tuples for every sentence in the file

    # Open the file using UTF-8 encoding
    with open(path, encoding="utf-8") as f:  # UTF-8 needed for Devanagari text

        # Read the file one line at a time
        for line in f:  # Iterate lazily so large files (train has ~1M rows) aren't loaded as raw text all at once
            line = line.strip()  # Remove leading/trailing whitespace

            # Skip empty lines
            if line:  # Guards against trailing blank lines at end of file
                r = json.loads(line)  # Parse the JSON string into a Python dictionary

                # Store the words (tokens) and their corresponding NER labels
                rows.append((r["words"], r["ner"]))  # r["words"] is the token list, r["ner"] the aligned IOB2 tag list

    # Return all loaded sentences and their NER labels
    return rows  # List of (tokens, tags) pairs representing the whole split


# Load Hindi training, validation, and testing datasets
hi = {s: load(os.path.join(d, f"hi_{s}.json")) for s in ["train", "val", "test"]}  # Dict keyed by split name -> list of (tokens, tags)

# Print the number of sentences in each dataset split
for s, rows in hi.items():  # Iterate over train/val/test
    print(s, len(rows))  # Report sentence counts to confirm expected dataset sizes


train 985787
val 13460
test 867


In [ ]:
from collections import Counter  # Counter gives convenient frequency counting for NER tag labels

# Analyze NER labels and data quality for each dataset split
for s, rows in hi.items():  # Loop over train/val/test splits
    tags = Counter()  # Count occurrences of each NER tag
    mismatch = 0  # Count sentences where words and tags have different lengths
    empty = 0  # Count empty sentences

    for w, n in rows:  # w = token list, n = NER tag list for one sentence
        tags.update(n)  # Update tag frequencies

        # Check whether the number of words matches the number of NER tags
        if len(w) != len(n):  # A mismatch would mean tokens and labels can't be zipped together safely
            mismatch += 1  # Increment the mismatch counter for this sentence

        # Check whether the sentence contains no words
        if len(w) == 0:  # Empty sentences are degenerate and may need to be filtered before training
            empty += 1  # Increment the empty-sentence counter

    # Print dataset statistics
    print(f"\n{'=' * 50}")  # Section divider for readability
    print(f"Dataset Split: {s.upper()}")  # Header naming the split being reported
    print(f"{'=' * 50}")  # Closing divider under the header
    print(f"Total sentences       : {len(rows):,}")  # Number of sentences in this split
    print(f"Total NER tags        : {sum(tags.values()):,}")  # Total tag occurrences across all sentences
    print(f"Length mismatches     : {mismatch:,}")  # Sentences where words/tags counts disagree
    print(f"Empty sentences       : {empty:,}")  # Sentences with zero tokens

    print("\nNER Tag Distribution:")  # Header for the per-tag breakdown below
    for tag, count in sorted(tags.items()):  # Iterate tags alphabetically for stable, readable output
        print(f"  {tag:<15}: {count:>10,}")  # Print each tag left-aligned with its right-aligned count

    print(f"{'=' * 50}")  # Trailing divider closing out this split's report



Dataset Split: TRAIN
Total sentences       : 985,787
Total NER tags        : 22,029,408
Length mismatches     : 0
Empty sentences       : 0

NER Tag Distribution:
  B-LOC          :    731,183
  B-ORG          :    686,388
  B-PER          :    767,003
  I-LOC          :    207,240
  I-ORG          :    825,713
  I-PER          :    712,403
  O              : 18,099,478

Dataset Split: VAL
Total sentences       : 13,460
Total NER tags        : 304,282
Length mismatches     : 0
Empty sentences       : 0

NER Tag Distribution:
  B-LOC          :     10,209
  B-ORG          :      9,735
  B-PER          :     10,549
  I-LOC          :      2,779
  I-ORG          :     11,506
  I-PER          :      9,870
  O              :    249,634

Dataset Split: TEST
Total sentences       : 867
Total NER tags        : 19,893
Length mismatches     : 0
Empty sentences       : 3

NER Tag Distribution:
  B-LOC          :        613
  B-ORG          :        521
  B-PER          :        788
  I-LOC      

In [ ]:
import numpy as np  # NumPy used for fast array math over sentence lengths

# Calculate the number of tokens in each training sentence
lens = np.array([len(w) for w, _ in hi["train"]])  # Build an array of per-sentence token counts for the train split

# Calculate and display sentence length statistics
print("\nTraining Dataset Sentence Length Statistics")  # Section header
print("-" * 45)  # Divider line
print(f"Minimum length  : {lens.min()}")  # Shortest sentence length in tokens
print(f"Median length   : {int(np.median(lens))}")  # Typical (50th percentile) sentence length
print(f"95th percentile : {int(np.percentile(lens, 95))}")  # Length covering 95% of sentences (useful for choosing max_length)
print(f"99th percentile : {int(np.percentile(lens, 99))}")  # Length covering 99% of sentences
print(f"Maximum length  : {lens.max()}")  # Longest sentence length in tokens
print("-" * 45)  # Closing divider



Training Dataset Sentence Length Statistics
---------------------------------------------
Minimum length  : 1
Median length   : 20
95th percentile : 48
99th percentile : 67
Maximum length  : 100
---------------------------------------------


In [ ]:
# Check whether the NER labels follow the IOB2 tagging scheme
def iob2_violations(rows):  # rows: list of (tokens, tags) pairs for one split
    bad = 0  # Count invalid I- tags

    # Iterate through each sentence and its corresponding NER tags
    for _, n in rows:  # Only the tag sequence n is needed here, tokens are ignored (_)
        prev = "O"  # Initialize previous tag as Outside

        for t in n:  # Walk the tag sequence left to right
            # An I- tag must follow a B- or I- tag of the same entity type
            if t.startswith("I-"):  # Only "inside" tags can be malformed under IOB2
                if prev == "O" or prev[2:] != t[2:]:  # Invalid if previous tag is O, or belongs to a different entity type
                    bad += 1  # Count this as an orphan/invalid I- tag

            # Update the previous tag
            prev = t  # Track the previous tag for the next iteration's check

    # Return the total number of IOB2 violations
    return bad  # Total count of malformed I- tags in this split


# Check IOB2 violations in training, validation, and test datasets
print("\nIOB2 Tagging Validation")  # Section header
print("-" * 45)  # Divider line

for s, rows in hi.items():  # Loop over train/val/test splits
    violations = iob2_violations(rows)  # Count orphan I- tags for this split
    print(f"{s.upper():<10} | Orphan I- tags: {violations:,}")  # Report the count per split

print("-" * 45)  # Closing divider



IOB2 Tagging Validation
---------------------------------------------
TRAIN      | Orphan I- tags: 5,513
VAL        | Orphan I- tags: 74
TEST       | Orphan I- tags: 7
---------------------------------------------


## Findings

Source: `ai4bharat/naamapadam`, Hindi splits, inspected directly from the distributed JSON files.

- Tag set: exactly `O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC` (IOB2). No other labels present.
- `words` and `ner` are always equal length (0 mismatches across all splits).
- Row counts: train 985,787 / val 13,460 / test 867.
- 3 empty rows in test, 0 in train/val.
- Word length: median 20, p95 48, p99 67, max 100.
- Orphan `I-` tags (continuation with no matching start): train 5,513, val 74, test 7 (~0.5%).

### Contract decisions

- Label noise: leave orphan `I-` tags as-is (keeps parity with the published benchmark; seqeval handles them).
- `max_length`: 128 subword tokens.
- Loader drops zero-length sentences on every split.
